# CS448 - Lab 3: Artificial Reverberation and Room Simulation


In this lab we will get some experience in simulating rooms digitally. We will design some simple reverbs using small filters and then work towards a more principled room simulator. You can use whichever sound you like for the following examples, usually a sound with sharp onsets is good for such experiments. One good example is the drum loop ```loop.wav```.

In [1]:
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
import scipy.signal as signal
# Make a sound player function that plays array "x" with a sample rate "rate", and labels it with "label"
def sound_player( x, rate=8000, label=''):
    from IPython.display import display, Audio, HTML
    display( HTML( 
    '<style> table, th, td {border: 0px; }</style> <table><tr><td>' + label + 
    '</td><td>' + Audio( x, rate=rate)._repr_html_()[3:] + '</td></tr></table>'
    ))

### Problem 1. Basic Room Simulation

The figure below is showing a room with mic and sound source locations. If you only care about the direct path (orange arrow), what would the filter look like? 

![alt text](room.png)

Based on this, add a few more early arrivals: there are four first reflections that hit the wall only once. I didn't draw those paths to tickle your brain. Reprsent them as peaks in your impulse response filter. Your filter must be with five peaks in total. Do the distance calculation correctly based on your grade school-level math skills!

You will need to make some assumptions. First of all, the speed of sound is 343 m/s. 
Plus, the amplitudes of the delayed impulses will get attenuated as the distance increases: it's inverse proportional to the distance. To this end, assume that the amplitude of the impulse, very close to the source (say 1cm), is 1. Based on this, calculate other ones in relation to it. Also, whenever the impulse hits the wall it also gets attenuated based on the reflection ratio of your choice, which will control the decaying of the peaks over time. Also, let's assume that the reflection ratio is 0.9 (i.e., a pretty reverberant room). Let's also assume that the sample rate is 16,000 Hz.

In [100]:
# The distances of the direct path and the first few reflections, and their attenuations (in samples)
# YOUR CODE HERE

# Attenuation of the delayed impulses, considering the distance and wall reflection
# YOUR CODE HERE

# Turn the attenuated and delayed impulses into a digital filter
# YOUR CODE HERE

# Plot the impulse response of the filter
# YOUR CODE HERE

### Problem 2. Pyroomacoustics

Enough about high school math! Now that we learned the basics, let's use a toolbox. Install ```pyroomacoustics``` package: ```pip install pyroomacoustics``` will be the easiest way to install it (https://github.com/LCAV/pyroomacoustics). Then, create a room of the same size shown in the previous figure (4 x 3 meters) by using the ```pra.ShoeBox``` function. It takes room dimensions, sample rate, and the mysterious ```materials``` parameter, which you can infer from the room size and RT60. As for RT60, just come up with a number, like 0.5 second. I gave you the code below for this part, so don't worry too much about it. 

Once you create the room, add the source and microphone to the room using the locations we used in the previous question. You will need to use ```add_microphone``` and ```add_source``` functions. Once all this is done, you are ready to create the RIR signal by using ```compute_rir```. There can be multiple RIRs by the number of mics and sources, but for this example, you have only one pair of source and mic, which will give you just one RIR signal. Check out the synthesized RIR and compare it with your analytic version above. Note that the RIR signal will be very long, so you want to truncate it and magnify to visualize only the beginning of the signal. Also note that there are about 40 samples of global delay that comes with the package's FIR filtering, so the filter samples are delayed by that amount (comared to your analytic solution). It's not a big deal, just wanted to be clear. 

See more information about the toolbox here: https://pyroomacoustics.readthedocs.io/en/pypi-release/pyroomacoustics.room.html

In [101]:
import pyroomacoustics as pra

rt60 = 0.5  # seconds
room_dim = [4, 3]  # In meters. The room is 4m wide and 3m long. Height is irrelevant for 2D simulations.

# We invert Sabine's formula to obtain the parameters for the ISM simulator
e_absorption, max_order = pra.inverse_sabine(rt60, room_dim, c=343)

# Create the room
room = pra.ShoeBox(
    room_dim, fs=48000, materials=pra.Material(e_absorption), max_order=max_order
)

# Add a microphone and a source, and compute the RIR
# Check out the waveform of the RIR
# YOUR CODE HERE


You can increase or decrease RT60 values to control the amount of reverberation. Play around with this number to come up with different RIRs. Then, do ```np.convolve``` to check out the reverberant versions of the ```loop.wav``` signal. 

In [102]:
# Try out the reverberant sound by convolving the RIR with an audio signal
# YOUR CODE HERE


# Now, try a few different rt60 values; repeat the process and check out the sound. 
# YOUR CODE HERE


### Problem 3. Measuring the room

Load the ```chirp.wav``` and ```chirp_reverb.wav``` and use them to reverse engineer the process. This time, you are not given the RIR signal, but you only have the dry source (chirp) and its reverby version (chirp_reverb). See slide 26-30 for more details, but basically, you will do the following

1. Convert the reverberant version and the input chirp using ```np.fft.fft```
2. Apply the equation in slide 28 to recover the RIR.
    - The equation in 28 is actually incorrect for this example, because the probe signal is not guaranteed to make sure the condition $X(\omega)X^*(\omega)\approx 1$ (the condition shown in slide 27). Instead, $X(\omega)X^*(\omega)=C(\omega)$ and $C(\omega)$ will be a large number. 
    - Hence, after you apply the equation in slide 28, you need to make sure that the result is scaled down properly, i.e., by dividing the estimated filter in frequency $H(\omega)$ by the power of the filter coefficient $X(\omega)X^*(\omega)=|X(\omega)|^2=C(\omega)$. That's the only difference. 
3. Convert the estimated filter in the frequency domain back to the time domain. Plot it. It should look familiar to you. 


In [103]:
# load the two audio files, chirp.wav and chirp_reverb.wav
# YOUR CODE HERE

# Do FFT-based deconvolution to estimate the impulse response of the room, and plot it
# Make sure the condition is met by dividing by the power of the FFT coefficient
# Plot the estimated IR. It should look familiar. 
# YOUR CODE HERE
